<a href="https://colab.research.google.com/github/ZinahO/AAI-520-Group-6-Multi-Agent-Financial-Analysis/blob/noslinn-routing-specialists/routing_specialist_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AAI-520 Final Team Project: Multi-Agent Financial Analysis System

## Routing, Specialist Agents, and Dynamic Tool Use

This section implements the routing workflow for the Investment
Research Agent. The router receives structured financial news from the
prompt-chaining workflow and directs each item to the Earnings, News, or
Market Specialist. The specialist agents then analyze the routed
information using appropriate financial tools and produce a structured
draft analysis for the Evaluator–Optimizer workflow.

In [ ]:
!pip install -q yfinance google-genai

In [ ]:
import json
import time

import pandas as pd
import yfinance as yf
from google import genai
from google.colab import userdata

In [3]:
gemini_api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=gemini_api_key,
    http_options={"timeout": 30000}
)

print("Gemini client connected successfully.")

Gemini client connected successfully.


In [4]:
last_llm_call = 0


def call_llm(prompt):
    """Send a prompt to Gemini with basic rate-limit handling."""
    global last_llm_call

    elapsed = time.time() - last_llm_call

    if elapsed < 15:
        time.sleep(15 - elapsed)

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            last_llm_call = time.time()
            return response.text

        except Exception:
            if attempt < 2:
                time.sleep(15)
            else:
                raise